In [2]:
import os
import torch
from pymilvus import connections, Collection
from transformers import AutoModel, AutoTokenizer
import ollama

# ========================
# CONFIGURAÇÃO DO MILVUS
# ========================
connections.connect("default", host="127.0.0.1", port="19530")
COLLECTION_NAME = "rag_embeddings_milvus"
collection = Collection(COLLECTION_NAME)

# ========================
# EMBEDDINGS (mesmo modelo usado na ingestão)
# ========================
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy().tolist()

# ========================
# FUNÇÃO DE RECUPERAÇÃO DO CONTEXTO
# ========================
def retrieve_context(query: str, top_k: int = 15):
    query_emb = get_embedding(query)

    collection.load()
    results = collection.search(
        data=[query_emb],
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["source_file", "source_url", "chunk_index", "chunk_text"]
    )

    contexts = []
    refs = []
    for r in results[0]:
        chunk_text = r.entity.get("chunk_text")
        source = r.entity.get("source_file")
        url = r.entity.get("source_url")
        contexts.append(chunk_text)
        refs.append(f"📄 {source} | 🔗 {url}")

    return "\n\n".join(contexts), "\n".join(refs)
    
def retrieve_images(query: str, top_k: int = 5):
    query_emb = get_embedding(query)

    image_collection = Collection("image_descriptions")
    image_collection.load()

    results = image_collection.search(
        data=[query_emb],
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},  # <- corrigido aqui
        limit=top_k,
        output_fields=["id", "url", "category", "titles", "texts"]
    )

    images = []
    for r in results[0]:
        url = r.entity.get("url")
        titles = r.entity.get("titles")
        texts = r.entity.get("texts")
        category = r.entity.get("category")
        score = r.distance
        images.append(
            f"🖼️ {category} | {titles} | {texts[:80]}... ({url}) [score={score:.3f}]"
        )

    return "\n".join(images)



# ========================
# FUNÇÃO DE GERAÇÃO DE RESPOSTA (via Ollama + Mistral)
# ========================
def generate_answer(query: str, context: str):
    prompt = f"""
Você é um assistente técnico especializado em licenciamento ambiental (EIA/RIMA).
Responda à pergunta do usuário **usando apenas o contexto fornecido**.

Contexto:
{context}

Pergunta:
{query}

Responda de forma clara, objetiva e técnica.
"""
    response = ollama.chat(
        model="mistral:7b",
        messages=[
            {"role": "system", "content": "Você é um assistente técnico ambiental especializado em EIA/RIMA."},
            {"role": "user", "content": prompt}
        ]
    )
    return response["message"]["content"]

# ========================
# LOOP DE CHAT
# ========================
print("🤖 Chatbot EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.\n")

while True:
    user_input = input("Você: ")
    if user_input.lower() in ["sair", "exit", "quit"]:
        break

    # Recupera contexto textual
    context, refs = retrieve_context(user_input)

    # Recupera imagens relacionadas
    images = retrieve_images(user_input)

    # Gera resposta
    answer = generate_answer(user_input, context)

    print("\nBot:", answer)
    print("\n--- Fontes ---")
    print(refs)

    if images:
        print("\n--- Imagens relacionadas ---")
        print(images)
    print("contexto: ")
    print (context)


🤖 Chatbot EIA/RIMA usando Milvus + Mistral 7B (Ollama). Digite 'sair' para encerrar.



Você:  Para a duplicação da SP-97, a equipe necessita de um mapa que traga informações relativas as bacias hidrográficas do estado de São Paulo. Necessitamos desse mapa para fazer um estudo de como procederemos quanto a permeabilidade do solo, conseguiria fornecer também informações da regulação vigente relativo a permeabilidade do solo?



Bot:  Para o projeto de duplicação da SP-97, o mapa necessário é o Mapa de Drenagem, que está disponível na AID GEO (Infraestruturas e Serviços Públicos). Este mapa apresenta as bacias hidrográficas do estado de São Paulo e fornecerá informações necessárias quanto à permeabilidade do solo.

Por outro lado, em relação à regulação vigente sobre a permeabilidade do solo no Estado de São Paulo, as regras estão abordadas na ADA GEO (Uso e Ocupação do Solo) e na AII GEO (Infraestruturas Existentes), porém não existe um mapa específico nessas fontes para a permeabilidade do solo. Para obter mais informações detalhadas, é recomendável consultar as fontes científicas, como o trabalho de Statterfield et al. (1998) chamado "Results from Quantitative Surveys In Brazil", que trata sobre medidas e monitoramento da biodiversidade no Brasil.

--- Fontes ---
📄 EIA-104-25-32165-25-Nova-Lig-Rod-Planalto-Bx-Sts-Ecovias.pdf | 🔗 https://cetesb.sp.gov.br/eiarima/eia/EIA-104-25-32165-25-Nova-Lig-Rod-Planalto

Você:  quais árvores existem em são paulo?



Bot:  As árvores encontradas durante o levantamento da flora na área estudada no estado de São Paulo incluem:

1. Alchornea sidifolia (tapiá-peludo)
2. Alchornea triplinervia (tapiá-mirim)
3. Andira fraxinifolia (angelim-amargoso, angelim-doce, jacarandá-do-mato)
4. Casearia sylvestris (guaçatonga)
5. Chusquea cf. bambusoides (criciúma)
6. Cupania oblongifolia (camboatá) e C. vernalis (camboatá-vermelho)
7. Dahlstedtia muehlbergiana (embira-de-sapo)
8. Ficus enormis (figueira)
9. Guarea kunthiana (marinheiro)
10. Hedyosmum brasiliense (chá-de-bugre, chá-de-índio, cidreira-do-mato, erva-de-soldado)
11. Jacaranda puberula (carobinha)
12. Lamanonia ternata (cibolo-amarelo)
13. Machaerium hirtum (jacarandá-de-espinhos)
14. Machaerium villosum (jacarandá-paulista)
15. Myroxylon peruiferum (cabreúva-vermelha)
16. Ocotea puberula (canela-branca)
17. Pterocarpus violaceus (pau-cigarra, caquera)
18. Tetrorchidium rubrivenium (canemuçu)
19. Tapirira guianensis (pau-pombo)
20. Vernonanthura poly

Você:  sair
